In [2]:
# ======================================================
# 1. Import Libraries
# ======================================================

import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# ======================================================
# 2. Load Dataset
# ======================================================

df = pd.read_excel("sales.xlsx")

print("First 5 Rows:")
print(df.head())


# ======================================================
# 3. Clean Column Names
# ======================================================

df.columns = df.columns.str.strip().str.lower()

print("\nColumns:")
print(df.columns)


# ======================================================
# 4. Convert Date Column
# ======================================================

df["date"] = pd.to_datetime(df["date"])

df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day

df.drop("date", axis=1, inplace=True)


# ======================================================
# 5. Encode Categorical Columns
# ======================================================

le_category = LabelEncoder()
le_segment = LabelEncoder()

df["product_category"] = le_category.fit_transform(df["product_category"])
df["customer_segment"] = le_segment.fit_transform(df["customer_segment"])


# ======================================================
# 6. Features & Target
# ======================================================

X = df[
    [
        "year",
        "month",
        "day",
        "product_category",
        "price",
        "discount",
        "customer_segment",
        "marketing_spend",
        "units_sold"
    ]
]

y = df["sales"]

print("\nTotal Features:", X.shape[1])


# ======================================================
# 7. Train Test Split
# ======================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)


# ======================================================
# 8. Train Models
# ======================================================

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)


rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)


# ======================================================
# 9. Evaluate Models
# ======================================================

print("\n----- Linear Regression -----")
print("R2:", r2_score(y_test, y_pred_lr))
print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("MSE:", mean_squared_error(y_test, y_pred_lr))


print("\n----- Random Forest -----")
print("R2:", r2_score(y_test, y_pred_rf))
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("MSE:", mean_squared_error(y_test, y_pred_rf))


# ======================================================
# 10. Choose Best Model
# ======================================================

if r2_score(y_test, y_pred_rf) > r2_score(y_test, y_pred_lr):

    best_model = rf
    print("\nRandom Forest performs better")

else:

    best_model = lr
    print("\nLinear Regression performs better")


# ======================================================
# 11. Save Model
# ======================================================

pickle.dump(best_model, open("model.pkl", "wb"))

print("\nModel saved successfully")


# ======================================================
# 12. Save Encoders
# ======================================================

pickle.dump(le_category, open("category_encoder.pkl", "wb"))
pickle.dump(le_segment, open("segment_encoder.pkl", "wb"))

print("Encoders saved successfully")


# ======================================================
# 13. Test Prediction
# ======================================================

sample = X.iloc[0].values.reshape(1, -1)

prediction = best_model.predict(sample)

print("\nSample Prediction:", prediction[0])

First 5 Rows:
        Date Product_Category   Price  Discount Customer_Segment  \
0 2023-01-01           Sports  932.80     35.82       Occasional   
1 2023-01-02             Toys  569.48      3.60          Premium   
2 2023-01-03       Home Decor  699.68      3.56          Premium   
3 2023-01-04             Toys  923.27      0.61          Premium   
4 2023-01-05             Toys  710.17     47.83          Premium   

   Marketing_Spend  Units_Sold     sales  
0          6780.38          32  29849.60  
1          6807.56          16   9111.68  
2          3793.91          27  18891.36  
3          9422.75          29  26774.83  
4          1756.83          17  12072.89  

Columns:
Index(['date', 'product_category', 'price', 'discount', 'customer_segment',
       'marketing_spend', 'units_sold', 'sales'],
      dtype='object')

Total Features: 9

----- Linear Regression -----
R2: 0.9468557727378571
MAE: 1537.9544719683029
MSE: 4881445.260827225

----- Random Forest -----
R2: 0.99251557

c:\users\richa\appdata\local\programs\python\python38\lib\site-packages\sklearn\base.py:465: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
